# Clase 12 — Fundamentos de NLPAutor: Josef Rodriguez## Objetivos de la claseAl finalizar esta clase el estudiante será capaz de:- Entender el problema fundamental del **Natural Language Processing (NLP)**- Comprender cómo convertir texto en variables numéricas- Aplicar **Bag of Words**- Aplicar **TF‑IDF**- Entrenar un modelo de clasificación de texto- Visualizar el espacio vectorial de documentos- Interpretar resultados de modelos NLP---## ¿Qué es NLP?**Natural Language Processing (NLP)** es una rama de la inteligencia artificial que busca que las computadoras puedan:- comprender lenguaje humano- interpretar texto- generar lenguaje naturalAplicaciones reales:- análisis de sentimiento- detección de spam- clasificación de documentos- chatbots- motores de búsqueda- traducción automática- asistentes virtuales

## El problema fundamental del NLPLos modelos de Machine Learning **no entienden texto**.Los modelos solo pueden trabajar con **números**.Por ejemplo:Texto:"I love machine learning"El modelo necesita verlo como:[0.12, 0.03, 0.88, 0.44, ...]Por lo tanto el desafío central del NLP es:**Convertir texto en vectores numéricos.**---## Pipeline clásico de NLPTexto  ↓  Limpieza  ↓  Tokenización  ↓  Eliminación de stopwords  ↓  Vectorización  ↓  Modelo de Machine Learning

# DatasetUsaremos el dataset **SMS Spam Collection**.Este dataset contiene mensajes SMS clasificados como:- ham → mensaje normal- spam → mensaje publicitario o fraudulentoNúmero de registros: ~5574 mensajesEste dataset es ideal para enseñanza porque:- es pequeño- texto real- clasificación clara

In [ ]:
import pandas as pdimport numpy as npimport matplotlib.pyplot as pltimport seaborn as snsimport reimport nltkfrom nltk.tokenize import word_tokenizefrom nltk.corpus import stopwordsfrom sklearn.model_selection import train_test_splitfrom sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizerfrom sklearn.linear_model import LogisticRegressionfrom sklearn.metrics import classification_report, confusion_matrix, accuracy_scorefrom sklearn.decomposition import PCAfrom sklearn.manifold import TSNEfrom wordcloud import WordCloud

# Cargar dataset

In [ ]:
df = pd.read_csv("SMSSpamCollection", sep="\t", header=None, names=["label","text"])df.head()

# Exploración de datos (EDA)Antes de modelar siempre debemos entender:- tamaño del dataset- distribución de clases- longitud de documentos

In [ ]:
df.shape

In [ ]:
sns.countplot(x=df["label"])plt.title("Distribución de clases")plt.show()

# Longitud de mensajesLos mensajes pueden tener longitudes muy diferentes.Esto es importante porque:- documentos largos tienen más palabras- documentos cortos tienen menos información

In [ ]:
df["length"] = df["text"].apply(len)plt.hist(df["length"], bins=40)plt.title("Distribución de longitud de mensajes")plt.show()

# Limpieza de textoProblemas comunes en texto:- mayúsculas- puntuación- símbolos- ruidoEjemplo:"WIN MONEY NOW!!!"Después de limpiar:"win money now"

In [ ]:
nltk.download("punkt")nltk.download("stopwords")stop_words = set(stopwords.words("english"))def clean_text(text):    text = text.lower()    text = re.sub(r"[^a-zA-Z\s]", "", text)    tokens = word_tokenize(text)    tokens = [w for w in tokens if w not in stop_words]    return " ".join(tokens)df["clean_text"] = df["text"].apply(clean_text)df.head()

# WordCloudUn WordCloud permite visualizar las palabras más frecuentes.

In [ ]:
spam_words = " ".join(df[df["label"]=="spam"]["clean_text"])wc = WordCloud(width=800,height=400,background_color="white").generate(spam_words)plt.imshow(wc)plt.axis("off")plt.title("WordCloud Spam")plt.show()

# Codificación de etiquetasLos modelos necesitan números.

In [ ]:
df["label_num"] = df["label"].map({"ham":0,"spam":1})

# Bag of WordsBag of Words representa un documento como un vector de frecuencias de palabras.Ejemplo:Documento 1: "I love ML"Documento 2: "I love AI"Vocabulario:[I, love, ML, AI]Representación:Doc1 → [1,1,1,0]Doc2 → [1,1,0,1]

In [ ]:
bow = CountVectorizer(max_features=3000)X_bow = bow.fit_transform(df["clean_text"])X_bow.shape

# TF‑IDFTF‑IDF mejora Bag of Words.La idea:Una palabra es importante si:- aparece mucho en un documento- aparece poco en otros documentosFormula:TFIDF = TF × IDF

In [ ]:
tfidf = TfidfVectorizer(max_features=3000)X_tfidf = tfidf.fit_transform(df["clean_text"])

# Train Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(    X_tfidf,    df["label_num"],    test_size=0.2,    random_state=42)

# Modelo de clasificación

In [ ]:
model = LogisticRegression()model.fit(X_train, y_train)

# Evaluación del modelo

In [ ]:
y_pred = model.predict(X_test)print(classification_report(y_test,y_pred))

# Matriz de confusión

In [ ]:
cm = confusion_matrix(y_test,y_pred)sns.heatmap(cm,annot=True,fmt="d")plt.title("Confusion Matrix")plt.show()

# Visualización del espacio TF‑IDF con PCACada documento es un vector de alta dimensión.PCA permite reducirlo a 2 dimensiones.

In [ ]:
pca = PCA(n_components=2)X_pca = pca.fit_transform(X_tfidf.toarray())plt.scatter(X_pca[:,0],X_pca[:,1],c=df["label_num"],alpha=0.5)plt.title("PCA representación documentos")plt.show()

# Visualización con t‑SNEt‑SNE es una técnica no lineal de reducción de dimensionalidad que preserva mejor los clusters.

In [ ]:
tsne = TSNE(n_components=2,perplexity=30,random_state=42)X_tsne = tsne.fit_transform(X_tfidf.toarray()[:2000])plt.scatter(X_tsne[:,0],X_tsne[:,1],c=df["label_num"][:2000],alpha=0.5)plt.title("t-SNE documentos")plt.show()

# Predicción en vivo

In [ ]:
def predict_sms(text):    clean = clean_text(text)    vec = tfidf.transform([clean])    pred = model.predict(vec)[0]    if pred==1:        return "SPAM"    else:        return "HAM"predict_sms("Congratulations you won a free ticket")